# Adaptive IDS Reinforcement Learning Notebook (Restructured)

This restructured notebook presents a clearer training workflow with refined separation of concerns. New cell order:

1. Overview (this cell)
2. Configuration & Environment
3. General Utilities (seeding, directory helpers)
4. Preprocessing (two-pass CSV -> memmap)
5. Dataset & Splits (creates DataLoaders)
6. Models (Dueling DQN + Actor-Critic)
7. Replay Buffer & Sampling (DQN specific)
8. Metrics & Evaluation Helpers
9. Training Routines (DQN & A3C)
10. Runner (orchestrates end-to-end pipeline)
11. Usage Guidance & Next Steps

Key Improvements:
- Explicit separation of evaluation helpers from training
- Replay buffer isolated (easier to swap with prioritized)
- Dataset splitting done before model definitions for clarity
- Autocast / AMP compatibility wrapper unified
- Reduced cross-cell implicit dependencies

Run cells in order. Modify only the Config cell then re-run the Runner for subsequent experiments.


In [1]:
# Cell 2: Configuration & Environment - STABILIZED MULTI-CLASS

import os
import dataclasses
from dataclasses import dataclass, asdict
from typing import Optional, List, Tuple
from contextlib import nullcontext
import numpy as np
import pandas as pd
import time
import itertools
import json
import matplotlib.pyplot as plt
import pickle

# PYTORCH & ML
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

# CONSTANTS
SEED_DEFAULT = 2024
DROP_PATTERNS = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'Source Port', 'Destination Port']
POSSIBLE_LABEL_COLS = ['Label', 'label', ' Label']


@dataclass
class Config:
    # MULTI-CLASS ARCHITECTURE (STABILIZED)
    binary: bool = False
    hidden_dims: str = '256,128'  # Reduced for stability
    epochs: int = 20
    batch_size: int = 64  # Reduced for stability
    lr: float = 0.0001  # Lower learning rate
    gamma: float = 0.99
    eps_start: float = 0.8
    eps_end: float = 0.1
    eps_decay_steps: int = 5000
    
    # STABILIZED LOSS & OPTIMIZATION
    focal_loss: bool = False  # Disabled for stability
    focal_alpha: float = 0.25
    focal_gamma: float = 1.5  # Reduced gamma
    use_class_weights: bool = True
    weight_clip: float = 100.0  # NEW: Clip extreme weights
    
    # RARE CLASS DETECTION (STABILIZED)
    rare_class_threshold: int = 1000
    r_tp_benign: float = 1.0
    r_tp_attack_base: float = 2.0
    r_tp_attack_rare: float = 5.0  # Reduced from 10.0
    r_fp: float = -1.0
    r_fn: float = -2.0
    
    # TRAINING STABILITY
    grad_clip: float = 1.0
    replay_size: int = 20_000  # Reduced from 60K
    target_sync_freq: int = 100
    early_stop_patience: int = 3
    
    # PATHS & MISC - ALL OUTPUTS ORGANIZED IN backend/model/
    data_dir: str = '../../data/2017'  # From backend/scripts/ -> root/data/
    output_dir: str = '../model/output'  # From backend/scripts/ -> backend/model/output/
    ckpt_dir: str = '../model/checkpoints'  # From backend/scripts/ -> backend/model/checkpoints/
    log_dir: str = '../model/logs'  # From backend/scripts/ -> backend/model/logs/
    cache_dir: str = '../model/cache'  # From backend/scripts/ -> backend/model/cache/
    compile: bool = False
    amp: bool = False
    device: Optional[str] = None
    low_mem: bool = False
    
    # Added missing fields for full pipeline compatibility
    num_workers: int = 0
    eval_fraction: float = 0.15
    val_fraction_of_train: float = 0.15
    seed: int = 2024
    dropout: float = 0.2
    algo: str = 'a3c'
    sequence: bool = False
    seq_len: int = 1
    weight_decay: float = 1e-4
    
    def parsed_hidden(self):
        return [int(x.strip()) for x in self.hidden_dims.split(',')]
    
    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.ckpt_dir, exist_ok=True)
        os.makedirs(self.log_dir, exist_ok=True)
        os.makedirs(self.cache_dir, exist_ok=True)

# Initialize stabilized configuration
cfg = Config()

# DEVICE SETUP
if cfg.device is None:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
else:
    device = torch.device(cfg.device)

if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name()
    total_mem = torch.cuda.get_device_properties(device).total_memory / 1e9
    print(f'[device] GPU: {gpu_name} ({total_mem:.1f}GB)')
else:
    print(f'[device] CPU: {torch.get_num_threads()} threads')

print(f'[config] 🎯 MULTI-CLASS ATTACK TYPE CLASSIFICATION (STABILIZED)')
print(f'[config] Model: {cfg.hidden_dims} hidden layers → multi-class outputs')
print(f'[config] Binary mode: {cfg.binary} (15 attack types)')
print(f'[config] Focal loss: {cfg.focal_loss}, Class weights: {cfg.use_class_weights}')
print(f'[config] Weight clipping: {cfg.weight_clip} (prevents NaN losses)')
print(f'[config] Batch size: {cfg.batch_size}, Learning rate: {cfg.lr}')
print(f'[config] Rare class bonus: {cfg.r_tp_attack_rare}x (threshold: {cfg.rare_class_threshold})')
print(f'[config] Training epochs: {cfg.epochs}, Early stop patience: {cfg.early_stop_patience}')
print(f'[config] Replay buffer: {cfg.replay_size:,} samples')
print(f'[config] Data directory: {cfg.data_dir}')
print(f'[config] Cache directory: {cfg.cache_dir}')
print(f'[config] Output directory: {cfg.output_dir}')
print(f'[config] Checkpoints directory: {cfg.ckpt_dir}')
print(f'[config] Logs directory: {cfg.log_dir}')
print()

# SEED FOR REPRODUCIBILITY  
torch.manual_seed(SEED_DEFAULT)
np.random.seed(SEED_DEFAULT)
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED_DEFAULT)

print(f'[seed] Set to {SEED_DEFAULT} for reproducible multi-class training')

[device] GPU: NVIDIA GeForce RTX 3050 Laptop GPU (4.3GB)
[config] 🎯 MULTI-CLASS ATTACK TYPE CLASSIFICATION (STABILIZED)
[config] Model: 256,128 hidden layers → multi-class outputs
[config] Binary mode: False (15 attack types)
[config] Focal loss: False, Class weights: True
[config] Weight clipping: 100.0 (prevents NaN losses)
[config] Batch size: 64, Learning rate: 0.0001
[config] Rare class bonus: 5.0x (threshold: 1000)
[config] Training epochs: 20, Early stop patience: 3
[config] Replay buffer: 20,000 samples
[config] Data directory: ../../data/2017
[config] Cache directory: ../model/cache
[config] Output directory: ../model/output
[config] Checkpoints directory: ../model/checkpoints
[config] Logs directory: ../model/logs

[seed] Set to 2024 for reproducible multi-class training


In [2]:
# Cell 3: General Utilities
POSSIBLE_LABEL_COLS = ['Label','label','labelname','LabelName','class','Class']
DROP_PATTERNS = ['id','timestamp','start_time','end_time','flow_id','unix_time','date']

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def sanitize_cols(cols):
    out=[]
    for c in cols:
        c2 = c.strip().replace(' ','_').replace('-','_').replace('/','_')
        c2 = ''.join(ch for ch in c2 if ord(ch)<128)
        out.append(c2)
    return out

def infer_label_col(cols):
    for name in POSSIBLE_LABEL_COLS:
        if name in cols:
            return name
    for c in cols:
        if c.lower()=='label':
            return c
    return None

def find_csvs(root_dir: str) -> List[str]:
    csvs = []
    for r,_,fs in os.walk(root_dir):
        for f in fs:
            if f.lower().endswith('.csv'):
                csvs.append(os.path.join(r,f))
    csvs.sort()
    return csvs

In [3]:
# Cell 4: Preprocessing (Two-pass CSV -> memmap)
# filepath: c:\AIML\Projects\adaptive-ids-v-2.0\backend\scripts\trailrl.ipynb

def two_pass_memmap(data_dir: str, cache_dir: str, binary: bool, chunksize: int=200_000, low_mem: bool=False, overwrite: bool=True):
    ensure_dir(cache_dir)
    # Use os.path.join for all path operations to ensure cross-platform compatibility
    X_path = os.path.join(cache_dir, 'X_mem.dat')
    y_path = os.path.join(cache_dir, 'y_mem.npy')
    meta_path = os.path.join(cache_dir, 'meta.pkl')
    if (not overwrite) and all(os.path.exists(p) for p in (X_path, y_path, meta_path)):
        print('[preprocess] Using existing memmap cache.')
        return X_path, y_path, meta_path

    print('[preprocess] scanning CSV files...')
    csvs = find_csvs(data_dir)
    if not csvs:
        raise FileNotFoundError(f'No CSV found under {data_dir}')
    print(f'[preprocess] found {len(csvs)} files')

    total_rows=0; feature_names=None; label_col=None
    sums=None; sumsqs=None; n_features=0; label_set=set()

    # First pass: gather stats
    for f in csvs:
        for i, chunk in enumerate(pd.read_csv(f, chunksize=chunksize, low_memory=False)):
            if i==0: print(f'[pass1] {os.path.basename(f)} ...')
            chunk.columns = sanitize_cols(list(chunk.columns))
            if label_col is None:
                label_col = infer_label_col(list(chunk.columns))
                if label_col is None: raise ValueError('Label column not found')
                print('[pass1] inferred label column:', label_col)
            drop_cols = [c for c in chunk.columns if any(p in c.lower() for p in DROP_PATTERNS)]
            if drop_cols: chunk = chunk.drop(columns=drop_cols, errors='ignore')
            if feature_names is None:
                numeric_cand=[]; temp = chunk.drop(columns=[label_col])
                for c in temp.columns:
                    coerced = pd.to_numeric(temp[c], errors='coerce')
                    if coerced.notna().sum()>0: numeric_cand.append(c)
                feature_names = numeric_cand; n_features=len(feature_names)
                sums = np.zeros(n_features, dtype=np.float64)
                sumsqs = np.zeros(n_features, dtype=np.float64)
                print(f'[pass1] detected {n_features} numeric features')
            arr = chunk[feature_names].apply(pd.to_numeric, errors='coerce').values.astype(np.float32)
            mask = np.isfinite(arr); arr[~mask] = np.nan
            sums += np.nan_to_num(np.nansum(arr, axis=0))
            sumsqs += np.nan_to_num(np.nansum(np.nan_to_num(arr)**2, axis=0))
            total_rows += arr.shape[0]
            lab = chunk[label_col].astype(str).str.strip().values
            label_set.update(lab.tolist())
            del chunk, arr
            if low_mem: gc.collect()
    if total_rows == 0: raise ValueError('No rows found after first pass')
    means = sums / max(1,total_rows)
    vars_ = (sumsqs / max(1,total_rows)) - means**2
    stds = np.sqrt(np.maximum(vars_, 1e-6))
    label_list = sorted(list(label_set))
    le = LabelEncoder(); le.fit(label_list)
    benign_vals = set()
    if binary:
        ben_mask = [l for l in label_list if l.lower().startswith('ben')]
        benign_vals = set(ben_mask) if ben_mask else set([l for l in label_list if 'benign' in l.lower()])
        le = LabelEncoder(); le.fit(['Benign','Attack'])
        print(f'[pass1] binary mode: benign labels mapped -> {benign_vals}')

    print('[pass1] rows:', total_rows, 'features:', n_features, 'unique labels:', len(label_list))

    # Prepare memmaps
    X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(total_rows, n_features))
    y_mm = np.zeros((total_rows,), dtype=np.int32)

    # Second pass: normalize + encode
    idx=0
    for f in csvs:
        for i, chunk in enumerate(pd.read_csv(f, chunksize=chunksize, low_memory=False)):
            if i==0: print(f'[pass2] {os.path.basename(f)} ...')
            chunk.columns = sanitize_cols(list(chunk.columns))
            chunk = chunk.drop(columns=[c for c in chunk.columns if any(p in c.lower() for p in DROP_PATTERNS)], errors='ignore')
            arr = chunk[feature_names].apply(pd.to_numeric, errors='coerce').values.astype(np.float32)
            inds = np.where(~np.isfinite(arr))
            if inds[0].size>0: arr[inds] = np.take(means.astype(np.float32), inds[1])
            arr = (arr - means.astype(np.float32)) / (stds.astype(np.float32)+1e-9)
            n = arr.shape[0]
            X_mm[idx:idx+n,:] = arr
            labs = chunk[infer_label_col(list(chunk.columns))].astype(str).str.strip().values
            if binary:
                mapped = np.array(['Benign' if l in benign_vals else 'Attack' for l in labs])
                y_enc = le.transform(mapped)
            else:
                y_enc = le.transform(labs)
            y_mm[idx:idx+n] = y_enc.astype(np.int32)
            idx += n
            del chunk, arr
            if low_mem: gc.collect()
    X_mm.flush(); np.save(y_path, y_mm)
    meta = {
        'n_features': n_features,
        'feature_names': feature_names,
        'means': means.astype(np.float32),
        'stds': stds.astype(np.float32),
        'label_classes': list(le.classes_),
        'total_rows': total_rows,
        'binary': binary
    }
    with open(meta_path,'wb') as f: pickle.dump(meta,f)
    print('[pass2] done. paths:', X_path, y_path)
    return X_path, y_path, meta_path

In [4]:
# Cell 5: Dataset & Stratified Splits
class MemmapDataset(Dataset):
    def __init__(self, X_path, y_path, meta_path, indices=None):
        self.meta = pickle.load(open(meta_path,'rb'))
        self.X_path = X_path
        self.y = np.load(y_path, mmap_mode='r')
        self.total = int(self.meta['total_rows'])
        self.n_features = int(self.meta['n_features'])
        self.X_mm = None
        self.indices = np.arange(self.total) if indices is None else np.array(indices, dtype=np.int64)
    def _ensure_open(self):
        if self.X_mm is None:
            self.X_mm = np.memmap(self.X_path, dtype='float32', mode='r', shape=(self.total, self.n_features))
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        i = int(self.indices[idx]); self._ensure_open()
        x = np.array(self.X_mm[i], dtype=np.float32, copy=True)
        y = int(self.y[i])
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

# Split helper

def create_splits(meta_path, y_path, cfg):
    meta = pickle.load(open(meta_path,'rb'))
    all_y = np.load(y_path, mmap_mode='r')
    idxs = np.arange(meta['total_rows'])
    sss = StratifiedShuffleSplit(n_splits=1, test_size=cfg.eval_fraction, random_state=cfg.seed)
    train_idx, test_idx = next(sss.split(idxs, all_y))
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=cfg.val_fraction_of_train, random_state=cfg.seed)
    train_idx, val_idx = next(sss2.split(train_idx, all_y[train_idx]))
    print('[split]', {'train': len(train_idx), 'val': len(val_idx), 'test': len(test_idx)})
    return meta, train_idx, val_idx, test_idx

In [5]:
# Cell 6: Models

def kaiming_init(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None: nn.init.zeros_(m.bias)

class DQN_MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dims=[256,128], dropout=0.2):
        super().__init__()
        feat=[]; last=input_dim
        for h in hidden_dims:
            feat += [nn.Linear(last,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last=h
        self.feature_net = nn.Sequential(*feat)
        self.advantage_net = nn.Sequential(nn.Linear(last,last//2), nn.ReLU(), nn.Linear(last//2, output_dim))
        self.value_net = nn.Sequential(nn.Linear(last,last//2), nn.ReLU(), nn.Linear(last//2,1))
        self.apply(kaiming_init)
    def forward(self,x):
        if x.dim()==3:
            B,S,F = x.shape; x = x.view(B,S*F)
        f = self.feature_net(x)
        adv = self.advantage_net(f)
        val = self.value_net(f)
        return val + (adv - adv.mean(dim=1, keepdim=True))

class ActorCritic(nn.Module):
    def __init__(self,input_dim,action_dim,hidden_dims=[256,128],use_lstm=False,seq_len=1,dropout=0.2):
        super().__init__(); self.use_lstm=use_lstm; self.seq_len=seq_len
        if use_lstm:
            self.feat = nn.Linear(input_dim, hidden_dims[0])
            self.lstm = nn.LSTM(hidden_dims[0], hidden_dims[1], batch_first=True)
            self.actor = nn.Linear(hidden_dims[1], action_dim)
            self.critic = nn.Linear(hidden_dims[1], 1)
        else:
            layers=[]; last=input_dim*(seq_len if seq_len>1 else 1)
            for h in hidden_dims:
                layers += [nn.Linear(last,h), nn.ReLU(), nn.Dropout(dropout)]
                last=h
            self.shared = nn.Sequential(*layers)
            self.actor = nn.Linear(last, action_dim)
            self.critic = nn.Linear(last,1)
        self.apply(kaiming_init)
    def forward(self,x):
        if self.use_lstm:
            h = F.relu(self.feat(x)); out,_ = self.lstm(h); last = out[:,-1,:]
            return self.actor(last), self.critic(last).squeeze(-1)
        else:
            if x.dim()==3:
                B,S,F = x.shape; x = x.view(B,S*F)
            h = self.shared(x)
            return self.actor(h), self.critic(h).squeeze(-1)

In [6]:
# Cell 8: Metrics & Evaluation Helpers - STABILIZED FOR MULTI-CLASS

import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance in multi-class classification"""
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

def compute_class_weights(y_train, label_classes, device, weight_clip=100.0):
    """Compute inverse frequency class weights with STABILITY IMPROVEMENTS"""
    y_train = np.array(y_train)
    unique_classes_in_data = np.unique(y_train)
    
    # Only compute weights for classes that actually exist in training data
    if len(unique_classes_in_data) < len(label_classes):
        print(f'[class_weights] Warning: Only {len(unique_classes_in_data)}/{len(label_classes)} classes present in training data')
        missing_classes = [label_classes[i] for i in range(len(label_classes)) if i not in unique_classes_in_data]
        print(f'[class_weights] Missing classes: {missing_classes}')
    
    # Compute weights only for present classes
    class_weights_computed = compute_class_weight('balanced', classes=unique_classes_in_data, y=y_train)
    
    # STABILITY FIX: Clip extreme weights to prevent NaN losses
    class_weights_computed = np.clip(class_weights_computed, 0.01, weight_clip)
    print(f'[class_weights] Applied clipping: [0.01, {weight_clip}] to prevent NaN losses')
    
    # Create full weight vector, assigning moderate weight to missing classes
    moderate_weight = np.median(class_weights_computed)  # Use median instead of mean
    class_weights_full = np.full(len(label_classes), moderate_weight, dtype=np.float32)
    
    # Fill in computed weights for present classes
    for i, class_idx in enumerate(unique_classes_in_data):
        class_weights_full[class_idx] = class_weights_computed[i]
    
    # Final stats
    print(f'[class_weights] Range: [{np.min(class_weights_full):.3f}, {np.max(class_weights_full):.3f}]')
    print(f'[class_weights] Ratio (max/min): {np.max(class_weights_full)/np.min(class_weights_full):.1f}x')
    
    return torch.FloatTensor(class_weights_full).to(device)

def compute_metrics(y_true, y_pred, label_classes):
    """Compute comprehensive metrics for multi-class classification"""
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'macro_recall': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'weighted_precision': float(precision_score(y_true, y_pred, average='weighted', zero_division=0)),
        'weighted_recall': float(recall_score(y_true, y_pred, average='weighted', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0))
    }

def save_confusion(y_true, y_pred, label_classes, output_dir, filename='confusion_matrix.png'):
    """Save confusion matrix with percentage normalization and proper labels"""
    cm = confusion_matrix(y_true, y_pred)
    
    # Normalize by row (true class) to show percentage of correct/incorrect predictions
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8) * 100
    
    plt.figure(figsize=(14, 12))
    
    # Use a better colormap for percentages
    im = plt.imshow(cm_norm, interpolation='nearest', cmap='Blues')
    plt.title(f'Multi-class Attack Type Classification\nConfusion Matrix (Row-normalized %)', fontsize=14, pad=20)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    
    # Set class labels
    tick_marks = np.arange(len(label_classes))
    plt.xticks(tick_marks, label_classes, rotation=45, ha='right')
    plt.yticks(tick_marks, label_classes)
    
    # Add percentage text annotations
    thresh = cm_norm.max() / 2.
    for i, j in itertools.product(range(cm_norm.shape[0]), range(cm_norm.shape[1])):
        color = "white" if cm_norm[i, j] > thresh else "black"
        plt.text(j, i, f'{cm_norm[i, j]:.1f}%\n({cm[i, j]})', 
                 horizontalalignment="center", verticalalignment="center", 
                 color=color, fontsize=8)
    
    plt.ylabel('True Attack Type', fontsize=12)
    plt.xlabel('Predicted Attack Type', fontsize=12)
    plt.tight_layout()
    
    conf_path = os.path.join(output_dir, filename)
    plt.savefig(conf_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    return conf_path

print('[metrics] 🎯 STABILIZED Multi-class evaluation functions loaded')
print('[metrics] Features: clipped class weights, stable focal loss, comprehensive metrics')
print('[metrics] STABILITY: Weight clipping prevents NaN losses in extreme imbalance scenarios')

[metrics] 🎯 STABILIZED Multi-class evaluation functions loaded
[metrics] Features: clipped class weights, stable focal loss, comprehensive metrics
[metrics] STABILITY: Weight clipping prevents NaN losses in extreme imbalance scenarios


In [7]:
# Cell 7: Replay Buffer (DQN)
class ReplayBuffer:
    def __init__(self, capacity:int, state_shape:Tuple[int]):
        self.capacity=int(capacity); self.state_shape=state_shape
        self.ptr=0; self.size=0
        self.states = np.zeros((self.capacity,*state_shape), dtype=np.float32)
        self.next_states = np.zeros_like(self.states)
        self.actions = np.zeros((self.capacity,), dtype=np.int64)
        self.rewards = np.zeros((self.capacity,), dtype=np.float32)
        self.dones = np.zeros((self.capacity,), dtype=np.uint8)
    def push_batch(self, states, actions, rewards, next_states, dones):
        for i in range(len(states)):
            self.states[self.ptr] = states[i]
            self.next_states[self.ptr] = next_states[i]
            self.actions[self.ptr] = int(actions[i])
            self.rewards[self.ptr] = float(rewards[i])
            self.dones[self.ptr] = 1 if dones[i] else 0
            self.ptr = (self.ptr+1) % self.capacity
            self.size = min(self.size+1, self.capacity)
    def sample(self, batch_size):
        idxs = np.random.randint(0, self.size, size=batch_size)
        return (torch.from_numpy(self.states[idxs]).float(),
                torch.from_numpy(self.actions[idxs]).long(),
                torch.from_numpy(self.rewards[idxs]).float(),
                torch.from_numpy(self.next_states[idxs]).float(),
                torch.from_numpy(self.dones[idxs]).float())

In [8]:
# Cell 9: STABILIZED Training Routines (DQN & A3C) - MULTI-CLASS ONLY

def _autocast_ctx(enabled, device):
    if not enabled: return nullcontext()
    dev_type = 'cuda' if device.type=='cuda' else 'cpu'
    try:
        return torch.amp.autocast(dev_type, enabled=True)
    except AttributeError:
        if dev_type=='cuda':
            return torch.cuda.amp.autocast(enabled=True)
        return nullcontext()

def maybe_compile(module, cfg):
    """Safely attempt torch.compile; return module unchanged on any failure or missing feature."""
    if not cfg.compile:
        return module
    try:
        import torch
        if not hasattr(torch, 'compile'):
            print('[compile] torch.compile unavailable in this version; skipping.')
            return module
        compiled = torch.compile(module)  # type: ignore[attr-defined]
        print('[compile] success')
        return compiled
    except Exception as e:
        print(f'[compile] skipped due to: {e}')
        return module

def train_dqn(model, target, optimizer, scheduler, replay, train_loader, val_loader, le, cfg, device, meta):
    """SIMPLIFIED DQN training for multi-class - PURE CLASSIFICATION"""
    writer = SummaryWriter(cfg.log_dir)
    
    # SIMPLIFIED: Pure classification loss only
    if cfg.use_class_weights:
        # Compute class weights from training data
        all_y_train = []
        for _, yb in train_loader:
            all_y_train.extend(yb.numpy())
        
        # Use stabilized class weights with clipping
        class_weights = compute_class_weights(all_y_train, le.classes_, device, cfg.weight_clip)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        print(f'[train] Using Cross-Entropy Loss with clipped class weights')
    else:
        criterion = nn.CrossEntropyLoss()
        print(f'[train] Using standard Cross-Entropy Loss')
    
    # Class statistics
    class_counts = {i: np.sum(np.array(all_y_train) == i) for i in range(len(le.classes_))}
    rare_classes = {i for i, count in class_counts.items() if count < cfg.rare_class_threshold}
    
    print(f'[train] Class counts: {[(le.classes_[i], class_counts[i]) for i in range(len(le.classes_))]}')
    print(f'[train] Rare classes: {[le.classes_[i] for i in rare_classes]}')
    
    best_score = -1.0
    best_epoch = 0

    # Safe compile
    model = maybe_compile(model, cfg)

    for epoch in range(cfg.epochs):
        model.train()
        losses = []
        
        for b, (xb, yb) in enumerate(train_loader):
            xb = xb.to(device)
            yb = yb.to(device)
            
            optimizer.zero_grad()
            
            with _autocast_ctx(cfg.amp, device):
                # Simple forward pass
                logits = model(xb)
                loss = criterion(logits, yb)
            
            # Check for NaN/Inf
            if torch.isnan(loss) or torch.isinf(loss):
                print(f'[train] Warning: Skipping NaN/Inf loss at epoch {epoch+1}, batch {b+1}')
                continue
            
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            
            if scheduler:
                scheduler.step()
            
            losses.append(loss.item())
        
        avg_loss = float(np.mean(losses)) if losses else 0.0
        print(f'[epoch {epoch+1}] loss={avg_loss:.5f}')
        writer.add_scalar('train/loss', avg_loss, epoch)

        # Validation
        model.eval()
        y_true = []
        y_pred = []
        
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                logits = model(xb)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                y_true.extend(yb.numpy())
                y_pred.extend(preds.tolist())
        
        metrics = compute_metrics(np.array(y_true), np.array(y_pred), le.classes_)
        score = metrics['macro_f1']
        
        print(f'[val] Acc: {metrics["accuracy"]:.3f} | Macro F1: {score:.3f} | '
              f'Macro Prec: {metrics["macro_precision"]:.3f} | Macro Rec: {metrics["macro_recall"]:.3f}')
        
        writer.add_scalar('val/macro_f1', score, epoch)
        writer.add_scalar('val/accuracy', metrics['accuracy'], epoch)
        
        if score > best_score:
            best_score = score
            best_epoch = epoch
            torch.save({
                'model_state_dict': model.state_dict(), 
                'meta': meta, 
                'label_classes': list(le.classes_), 
                'cfg': asdict(cfg)
            }, os.path.join(cfg.ckpt_dir, 'best_model.pth'))
            print('[ckpt] best_model.pth saved')
        elif (epoch - best_epoch) >= cfg.early_stop_patience:
            print('[early-stop] patience reached')
            break
    
    torch.save({
        'model_state_dict': model.state_dict(), 
        'meta': meta, 
        'label_classes': list(le.classes_), 
        'cfg': asdict(cfg)
    }, os.path.join(cfg.ckpt_dir, 'final_model.pth'))
    writer.close()
    return model

def train_a3c(model, optimizer, train_loader, val_loader, le, cfg, device, meta):
    """A3C training for multi-class classification"""
    writer = SummaryWriter(cfg.log_dir)
    
    # Classification loss
    if cfg.use_class_weights:
        all_y_train = []
        for _, yb in train_loader:
            all_y_train.extend(yb.numpy())
        
        class_weights = compute_class_weights(all_y_train, le.classes_, device, cfg.weight_clip)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        print('[train] Using Cross-Entropy Loss with class weights for A3C')
    else:
        criterion = nn.CrossEntropyLoss()
        print('[train] Using standard Cross-Entropy Loss for A3C')
    
    best_score = -1.0
    best_epoch = 0
    
    model = maybe_compile(model, cfg)
    
    for epoch in range(cfg.epochs):
        model.train()
        losses = []
        
        for b, (xb, yb) in enumerate(train_loader):
            xb = xb.to(device)
            yb = yb.to(device)
            
            optimizer.zero_grad()
            
            with _autocast_ctx(cfg.amp, device):
                # Actor-Critic forward pass
                action_logits, values = model(xb)
                
                # Classification loss (treat as action selection)
                actor_loss = criterion(action_logits, yb)
                
                # Critic loss (predict classification confidence)
                target_values = torch.ones_like(values) * 0.5  # Neutral baseline
                critic_loss = F.mse_loss(values, target_values)
                
                total_loss = actor_loss + 0.5 * critic_loss
            
            if torch.isnan(total_loss) or torch.isinf(total_loss):
                print(f'[train] Warning: Skipping NaN/Inf loss at epoch {epoch+1}, batch {b+1}')
                continue
            
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            
            losses.append(total_loss.item())
        
        avg_loss = float(np.mean(losses)) if losses else 0.0
        print(f'[epoch {epoch+1}] loss={avg_loss:.5f}')
        writer.add_scalar('train/loss', avg_loss, epoch)
        
        # Validation
        model.eval()
        y_true = []
        y_pred = []
        
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                action_logits, _ = model(xb)
                preds = torch.argmax(action_logits, dim=1).cpu().numpy()
                y_true.extend(yb.numpy())
                y_pred.extend(preds.tolist())
        
        metrics = compute_metrics(np.array(y_true), np.array(y_pred), le.classes_)
        score = metrics['macro_f1']
        
        print(f'[val] Acc: {metrics["accuracy"]:.3f} | Macro F1: {score:.3f} | '
              f'Macro Prec: {metrics["macro_precision"]:.3f} | Macro Rec: {metrics["macro_recall"]:.3f}')
        
        writer.add_scalar('val/macro_f1', score, epoch)
        writer.add_scalar('val/accuracy', metrics['accuracy'], epoch)
        
        if score > best_score:
            best_score = score
            best_epoch = epoch
            torch.save({
                'model_state_dict': model.state_dict(), 
                'meta': meta, 
                'label_classes': list(le.classes_), 
                'cfg': asdict(cfg)
            }, os.path.join(cfg.ckpt_dir, 'best_model.pth'))
            print('[ckpt] best_model.pth saved')
        elif (epoch - best_epoch) >= cfg.early_stop_patience:
            print('[early-stop] patience reached')
            break
    
    torch.save({
        'model_state_dict': model.state_dict(), 
        'meta': meta, 
        'label_classes': list(le.classes_), 
        'cfg': asdict(cfg)
    }, os.path.join(cfg.ckpt_dir, 'final_model.pth'))
    writer.close()
    return model

print('[train]  SIMPLIFIED multi-class training functions loaded')
print('[train] Features: pure classification loss, NaN prevention, balanced evaluation')

[train]  SIMPLIFIED multi-class training functions loaded
[train] Features: pure classification loss, NaN prevention, balanced evaluation


In [9]:
# Cell: Final Artifacts & Evaluation

def final_artifacts(model, test_loader, le, cfg, device, meta):
    """
    Evaluate model on test set and save final artifacts.
    Returns comprehensive metrics for multi-class classification.
    """
    print('[eval] Running final evaluation on test set...')
    
    model.eval()
    y_true = []
    y_pred = []
    
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            
            if cfg.algo == 'dqn':
                logits = model(xb)
            else:  # a3c
                logits, _ = model(xb)
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            y_true.extend(yb.numpy())
            y_pred.extend(preds.tolist())
    
    # Compute final metrics
    metrics = compute_metrics(np.array(y_true), np.array(y_pred), le.classes_)
    
    print(f'[eval] Final Test Results:')
    print(f'       Accuracy: {metrics["accuracy"]:.3f}')
    print(f'       Macro F1: {metrics["macro_f1"]:.3f}')
    print(f'       Macro Precision: {metrics["macro_precision"]:.3f}')
    print(f'       Macro Recall: {metrics["macro_recall"]:.3f}')
    print(f'       Weighted F1: {metrics["weighted_f1"]:.3f}')
    
    # Save confusion matrix
    conf_path = save_confusion(np.array(y_true), np.array(y_pred), le.classes_, cfg.output_dir)
    print(f'[eval] Confusion matrix saved to: {conf_path}')
    
    # Save final artifacts
    import time
    run_id = f"run_{int(time.time())}"
    save_dir = os.path.join(cfg.output_dir, run_id)
    os.makedirs(save_dir, exist_ok=True)
    
    # Save final model
    model_path = os.path.join(save_dir, 'final_model.pt')
    torch.save(model.state_dict(), model_path)
    
    # Save metrics
    metrics_path = os.path.join(save_dir, 'test_metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    # Save config
    config_path = os.path.join(save_dir, 'config.json')
    with open(config_path, 'w') as f:
        json.dump(dataclasses.asdict(cfg), f, indent=2)
    
    # Save label mapping
    label_path = os.path.join(save_dir, 'label_classes.json')
    with open(label_path, 'w') as f:
        json.dump(list(le.classes_), f, indent=2)
    
    print(f"[eval] Final artifacts saved to: {save_dir}")
    return metrics

In [10]:
# Cell 10: Runner (End-to-End) - OPTIMIZED FOR MULTI-CLASS

print('[runner] MULTI-CLASS ATTACK TYPE CLASSIFICATION PIPELINE')

if cfg.compile:
    print('[runner] cfg.compile is True; will attempt safe compile inside training routines.')
else:
    print('[runner] compile disabled (set cfg.compile=True in Config to attempt).')

# 1. Preprocess
print('[1/6] Preprocessing CSV files...')
X_path, y_path, meta_path = two_pass_memmap(
    cfg.data_dir,
    cfg.cache_dir,
    cfg.binary,
    chunksize=100_000 if not cfg.low_mem else 50_000,
    low_mem=cfg.low_mem,
    overwrite=True
)

meta = pickle.load(open(meta_path, 'rb'))
label_classes = meta['label_classes']
le = LabelEncoder()
le.fit(label_classes)

print(f'[runner] Number of classes: {len(label_classes)}')
print(f'[runner] Classes: {label_classes}')

# 2. Create splits
print('[2/6] Creating stratified splits...')
meta, train_idx, val_idx, test_idx = create_splits(meta_path, y_path, cfg)

# 3. Create datasets and loaders
print('[3/6] Creating data loaders...')
input_dim = meta['n_features']
ds_train = MemmapDataset(X_path, y_path, meta_path, indices=train_idx)
ds_val = MemmapDataset(X_path, y_path, meta_path, indices=val_idx)
ds_test = MemmapDataset(X_path, y_path, meta_path, indices=test_idx)

loader_args = dict(
    batch_size=cfg.batch_size, 
    num_workers=cfg.num_workers, 
    pin_memory=(device.type == 'cuda')
)
train_loader = DataLoader(ds_train, shuffle=True, **loader_args)
val_loader = DataLoader(ds_val, shuffle=False, **loader_args)
test_loader = DataLoader(ds_test, shuffle=False, **loader_args)

print(f'[data] Train: {len(ds_train)}, Val: {len(ds_val)}, Test: {len(ds_test)}')

# 4. Initialize model
print('[4/6] Initializing model...')
hidden_dims = cfg.parsed_hidden()
num_actions = len(le.classes_)

print(f'[model] Input features: {input_dim}')
print(f'[model] Output classes: {num_actions}')
print(f'[model] Hidden layers: {hidden_dims}')
print(f'[model] Algorithm: {cfg.algo.upper()}')

if cfg.algo == 'dqn':
    model = DQN_MLP(input_dim, num_actions, hidden_dims, cfg.dropout).to(device)
    target = DQN_MLP(input_dim, num_actions, hidden_dims, cfg.dropout).to(device)
    target.load_state_dict(model.state_dict())
    print('[model] DQN with target network initialized')
else:
    model = ActorCritic(
        input_dim, num_actions, hidden_dims, 
        use_lstm=cfg.sequence, seq_len=cfg.seq_len, dropout=cfg.dropout
    ).to(device)
    target = None
    print('[model] Actor-Critic initialized')

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=cfg.lr, 
    weight_decay=cfg.weight_decay
)

scheduler = None
if cfg.algo == 'dqn':
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.epochs * len(train_loader)
    )

# 5. Train
print('[5/6] Starting training...')
if cfg.algo == 'dqn':
    # Simplified DQN - no replay buffer, pure classification
    model = train_dqn(model, target, optimizer, scheduler, None, train_loader, val_loader, le, cfg, device, meta)
else:
    model = train_a3c(model, optimizer, train_loader, val_loader, le, cfg, device, meta)

# 6. Final evaluation
print('[6/6] Final evaluation...')
metrics = final_artifacts(model, test_loader, le, cfg, device, meta)

print(f'\n TRAINING COMPLETE!')
print(f'   Best Macro F1: {metrics["macro_f1"]:.3f}')
print(f'   Final Accuracy: {metrics["accuracy"]:.3f}')
print(f'   Model saved in: {cfg.ckpt_dir}')
print(f'   Logs saved in: {cfg.log_dir}')

[runner] MULTI-CLASS ATTACK TYPE CLASSIFICATION PIPELINE
[runner] compile disabled (set cfg.compile=True in Config to attempt).
[1/6] Preprocessing CSV files...
[preprocess] scanning CSV files...
[preprocess] found 8 files
[pass1] Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv ...
[pass1] inferred label column: Label
[pass1] detected 74 numeric features
[pass1] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv ...
[pass1] Friday-WorkingHours-Morning.pcap_ISCX.csv ...
[pass1] Monday-WorkingHours.pcap_ISCX.csv ...
[pass1] Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv ...
[pass1] Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv ...
[pass1] Tuesday-WorkingHours.pcap_ISCX.csv ...
[pass1] Wednesday-workingHours.pcap_ISCX.csv ...
[pass1] rows: 2830743 features: 74 unique labels: 15
[pass2] Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv ...
[pass2] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv ...
[pass2] Friday-WorkingHours-Morning.pcap_ISCX.csv ...
[pass2] M

# Notebook Documentation: Adaptive IDS Training Pipeline

## 1. Overview

This Jupyter Notebook implements a complete, end-to-end pipeline for training and evaluating a multi-class Intrusion Detection System (IDS). It treats the detection problem as a supervised classification task, leveraging concepts and architectures from Reinforcement Learning (like Dueling DQN and Actor-Critic) but adapted for pure classification.

The primary goal is to classify network traffic into multiple categories: one "Benign" class and several distinct "Attack" types. The pipeline is designed to be robust, handling large datasets efficiently and addressing common issues like class imbalance.

## 2. The Workflow

The notebook executes a sequential pipeline, orchestrated by the **Runner cell (Cell 10)**. Each step is a prerequisite for the next.

### Step 1: Configuration
- **What it is:** A centralized `Config` dataclass (Cell 1) holds all hyperparameters, file paths, and operational flags (e.g., `algo`, `epochs`, `lr`).
- **How it works:** The user modifies this single object to control the entire experiment. All subsequent cells read their configuration from the `cfg` instance.

### Step 2: Preprocessing
- **What it is:** A highly efficient, two-pass preprocessing step (`two_pass_memmap` function in Cell 3) converts raw CSV files into a standardized, machine-learning-ready format.
- **How it works:**
    1.  **First Pass:** The function scans all CSV files in the `data_dir` to gather metadata without loading everything into memory. It determines the feature set, calculates the mean and standard deviation for each feature, and identifies all unique class labels.
    2.  **Second Pass:** It re-reads the CSVs, normalizes the data using the calculated stats (Z-score normalization), encodes labels into integers, and writes the results to disk as memory-mapped files (`.dat` for features, `.npy` for labels). This allows the notebook to handle datasets larger than the available RAM.

### Step 3: Data Splitting & Loading
- **What it is:** The preprocessed data is split into training, validation, and test sets, and then loaded for the model.
- **How it works:**
    - `create_splits` (Cell 4) uses `StratifiedShuffleSplit` to ensure that the proportion of each attack class is maintained across the train, validation, and test sets. This is crucial for reliable evaluation on imbalanced data.
    - The `MemmapDataset` class (Cell 4) is a custom PyTorch `Dataset` that reads directly from the memory-mapped files on disk, providing efficient data access during training.
    - Standard PyTorch `DataLoader`s are created to handle batching, shuffling, and parallel data loading.

### Step 4: Model Initialization
- **What it is:** The neural network model is defined and initialized based on the configuration.
- **How it works:**
    - The `cfg.algo` parameter determines which model to use:
        - `'dqn'`: A `DQN_MLP` (Cell 5), which uses a Dueling Deep Q-Network architecture adapted for classification.
        - `'a3c'`: An `ActorCritic` model (Cell 5).
    - The model is moved to the appropriate `device` (GPU or CPU).
    - An `AdamW` optimizer is initialized to update the model's weights.

### Step 5: Model Training
- **What it is:** The core training loop where the model learns to classify network traffic.
- **How it works:**
    - The notebook uses a pure classification training routine (`train_dqn` or `train_a3c` in Cell 8).
    - **Loss Function:** It uses `CrossEntropyLoss`. To combat class imbalance, it can compute and apply class weights, which give more importance to under-represented attack types. These weights are clipped to prevent numerical instability (NaN losses).
    - **Training Loop:** The model iterates through the training data for a set number of `epochs`.
    - **Validation & Early Stopping:** After each epoch, the model's performance is measured on the validation set using the Macro F1 score. If the score does not improve for a specified number of epochs (`early_stop_patience`), training is halted to prevent overfitting.
    - **Checkpointing:** The best-performing model (based on validation score) is saved to disk (`best_model.pth`).

### Step 6: Final Evaluation & Artifact Generation
- **What it is:** The final, unbiased performance of the trained model is assessed on the unseen test set.
- **How it works:**
    - The `final_artifacts` function (Cell 9) loads the best model and runs it on the `test_loader`.
    - It computes a comprehensive set of metrics (Accuracy, Macro/Weighted Precision, Recall, F1).
    - It generates and saves a detailed confusion matrix plot, which helps visualize the model's performance on a per-class basis.
    - All final results—the model state, test metrics, configuration, and label mappings—are saved into a unique, timestamped directory in `output_dir` for full reproducibility.

| Metric                | Value   | Description                                             |
|-----------------------|---------|---------------------------------------------------------|
| r_tp_benign           | 1.0     | Reward for true positive (correctly classified Benign)  |
| r_tp_attack_base      | 2.0     | Reward for true positive (correctly classified Attack)  |
| r_tp_attack_rare      | 5.0     | Reward for true positive (rare Attack class)            |
| r_fp                  | -1.0    | Penalty for false positive (Benign misclassified)       |
| r_fn                  | -2.0    | Penalty for false negative (Attack missed)              |
| rare_class_threshold  | 1000    | Threshold for defining a rare class (sample count)      |

| Label                         | Description                                                                                  |
|-------------------------------|----------------------------------------------------------------------------------------------|
| BENIGN                        | Normal, non-malicious network traffic.                                                       |
| Bot                           | Traffic generated by a botnet, often for coordinated attacks like DDoS or spam.              |
| DDoS                          | Distributed Denial of Service; an attack overwhelming a target with traffic from multiple sources. |
| DoS GoldenEye                 | Application-layer DoS attack using fully-formed HTTP requests to exhaust server resources.   |
| DoS Hulk                      | Application-layer DoS attack that uses unique requests to bypass caching engines.            |
| DoS Slowhttptest              | A tool for launching slow DoS attacks (e.g., slow POST) to tie up server connections.        |
| DoS slowloris                 | A slow DoS attack that sends partial HTTP requests to exhaust server connection slots.       |
| FTP-Patator                   | A brute-force attack attempting to guess credentials for the FTP service.                    |
| Heartbleed                    | An attack exploiting a vulnerability in OpenSSL to steal sensitive data from server memory.  |
| Infiltration                  | Traffic indicating an attacker has gained internal access and is exploring or exfiltrating data. |
| PortScan                      | An attack that scans server ports to discover open and potentially vulnerable services.      |
| SSH-Patator                   | A brute-force attack attempting to guess credentials for the SSH service.                    |
| Web Attack – Brute Force      | Systematically trying all possible passwords to gain access to a web application.            |
| Web Attack – Sql Injection    | Injecting malicious SQL code into web inputs to manipulate the backend database.             |
| Web Attack – XSS              | Cross-Site Scripting; injecting malicious scripts into websites to be executed by other users. |